**STEP 1 — Data preparation and LULC statistics**

In [ ]:
#Install required packages
!pip -q install rasterio openpyxl pandas numpy matplotlib

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive

Libraries successfully imported.


In [ ]:
# Mount Google Drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Working directory
WORKDIR = "/content/drive/MyDrive/lulcmaps"

os.chdir(WORKDIR)

print("Working directory:")
print(os.getcwd())

Working directory:
/content/drive/MyDrive/lulcmaps


In [ ]:
# Input rasters
FILES = {

    2000: "LULC2000.tif",
    2010: "LULC2010.tif",
    2020: "LULC2020.tif"

}

In [ ]:
# LULC classes
CLASS_NAMES = {

    1: "Forest",
    2: "Rangeland",
    3: "Wetland",
    4: "Water body",
    5: "Cropland",
    6: "Built-up"

}

NODATA = 127

In [ ]:
# Read rasters
rasters = {}

profiles = {}

pixel_area = None

for year, filename in FILES.items():

    with rasterio.open(filename) as src:

        rasters[year] = src.read(1)

        profiles[year] = src.profile

        transform = src.transform

        pixel_area = transform.a * abs(transform.e)

print("All rasters successfully loaded.")

All rasters successfully loaded.


In [ ]:
# Verify raster compatibility
shape = rasters[2000].shape
crs = profiles[2000]["crs"]

for year in [2010, 2020]:

    assert rasters[year].shape == shape, f"Shape mismatch for {year}"

    assert profiles[year]["crs"] == crs, f"CRS mismatch for {year}"

print("✓ Raster dimensions are identical.")

print("✓ Coordinate reference systems are identical.")

print(f"Raster size : {shape[0]} rows × {shape[1]} columns")

print(f"Pixel area : {pixel_area:.0f} m²")

✓ Raster dimensions are identical.
✓ Coordinate reference systems are identical.
Raster size : 6932 rows × 6855 columns
Pixel area : 900 m²


In [ ]:
# Calculate LULC statistics
def calculate_statistics(raster):

    valid = raster[raster != NODATA]

    total_pixels = len(valid)

    results = []

    for cls in CLASS_NAMES:

        pixels = np.sum(valid == cls)

        area = pixels * pixel_area / 1e6

        percentage = pixels / total_pixels * 100

        results.append({

            "Class": cls,

            "Land use class": CLASS_NAMES[cls],

            "Pixels": pixels,

            "Area (km²)": round(area,2),

            "Area (%)": round(percentage,2)

        })

    return pd.DataFrame(results)

In [ ]:
# Calculate statistics for all years
statistics = {}

for year in FILES:

    statistics[year] = calculate_statistics(rasters[year])

statistics[2000]

,Class,Land use class,Pixels,Area (km²),Area (%)
0,1,Forest,4803110,4322.80,31.75
1,2,Rangeland,6421049,5778.94,42.45
2,3,Wetland,204864,184.38,1.35
3,4,Water body,1950901,1755.81,12.90
4,5,Cropland,1009715,908.74,6.68
5,6,Built-up,736487,662.84,4.87


In [ ]:
# Summary table
summary = pd.DataFrame({

    "Land use class": statistics[2000]["Land use class"],

    "2000 (km²)": statistics[2000]["Area (km²)"],

    "2010 (km²)": statistics[2010]["Area (km²)"],

    "2020 (km²)": statistics[2020]["Area (km²)"],

    "2000 (%)": statistics[2000]["Area (%)"],

    "2010 (%)": statistics[2010]["Area (%)"],

    "2020 (%)": statistics[2020]["Area (%)"]

})

summary

,Land use class,2000 (km²),2010 (km²),2020 (km²),2000 (%),2010 (%),2020 (%)
0,Forest,4322.80,4239.99,4117.23,31.75,31.15,30.24
1,Rangeland,5778.94,5559.32,5018.79,42.45,40.84,36.87
2,Wetland,184.38,177.79,150.02,1.35,1.31,1.10
3,Water body,1755.81,1752.21,1765.01,12.90,12.87,12.97
4,Cropland,908.74,1125.69,1577.83,6.68,8.27,11.59
5,Built-up,662.84,758.51,984.63,4.87,5.57,7.23


In [ ]:
# Export statistics
summary.to_csv("LULC_Area_Statistics_2000_2020.csv", index=False)

summary.to_excel("LULC_Area_Statistics_2000_2020.xlsx", index=False)

✓ Statistics exported successfully.


**STEP 2 — Land use transition analysis**

In [ ]:
# Transition matrix function
def transition_matrix(map_from, map_to):

    valid = (map_from != NODATA) & (map_to != NODATA)

    from_pixels = map_from[valid]
    to_pixels = map_to[valid]

    matrix = np.zeros((6,6), dtype=np.int64)

    for i in range(1,7):
        for j in range(1,7):

            matrix[i-1, j-1] = np.sum(
                (from_pixels == i) &
                (to_pixels == j)
            )

    matrix = matrix * pixel_area / 1e6

    matrix = pd.DataFrame(
        matrix,
        index=list(CLASS_NAMES.values()),
        columns=list(CLASS_NAMES.values())
    )

    return matrix.round(2)

In [ ]:
# Compute transition matrices
TM_2000_2010 = transition_matrix(
    rasters[2000],
    rasters[2010]
)

TM_2010_2020 = transition_matrix(
    rasters[2010],
    rasters[2020]
)

TM_2000_2010

Transition matrices successfully computed.


,Forest,Rangeland,Wetland,Water body,Cropland,Built-up
Forest,4183.88,84.42,0.00,0.02,31.90,22.58
Rangeland,43.02,5417.26,0.00,0.28,253.54,64.85
Wetland,0.00,0.00,170.93,1.88,9.39,2.18
Water body,0.02,0.72,4.60,1750.01,0.00,0.45
Cropland,13.08,56.91,2.25,0.02,830.87,5.61
Built-up,0.00,0.00,0.00,0.00,0.00,662.84


In [ ]:
# Calculate persistence, gains and losses
def transition_statistics(matrix):

    persistence = np.diag(matrix)

    loss = matrix.sum(axis=1) - persistence

    gain = matrix.sum(axis=0) - persistence

    net = gain - loss

    stats = pd.DataFrame({

        "Persistence (km²)": persistence,

        "Loss (km²)": loss,

        "Gain (km²)": gain,

        "Net change (km²)": net

    }, index=matrix.index)

    return stats.round(2)

CHANGE_2000_2010 = transition_statistics(TM_2000_2010)

CHANGE_2010_2020 = transition_statistics(TM_2010_2020)

CHANGE_2000_2010

,Persistence (km²),Loss (km²),Gain (km²),Net change (km²)
Forest,4183.88,138.92,56.12,-82.80
Rangeland,5417.26,361.69,142.05,-219.64
Wetland,170.93,13.45,6.85,-6.60
Water body,1750.01,5.79,2.20,-3.59
Cropland,830.87,77.87,294.83,216.96
Built-up,662.84,0.00,95.67,95.67


In [ ]:
# Prepare Sankey data
def create_sankey_table(matrix, year_from, year_to):

    rows = []

    for i, source_class in enumerate(matrix.index):

        for j, target_class in enumerate(matrix.columns):

            area = matrix.iloc[i, j]

            if area > 0:

                rows.append({

                    "source_year": year_from,

                    "target_year": year_to,

                    "source_class": source_class,

                    "target_class": target_class,

                    "area_km2": area

                })

    return pd.DataFrame(rows)

In [ ]:
# Create Sankey dataset
Sankey_2000_2010 = create_sankey_table(
    TM_2000_2010,
    2000,
    2010
)

Sankey_2010_2020 = create_sankey_table(
    TM_2010_2020,
    2010,
    2020
)

Sankey_Data = pd.concat(
    [Sankey_2000_2010,
     Sankey_2010_2020],
    ignore_index=True
)

Sankey_Data.head()

,source_year,target_year,source_class,target_class,area_km2
0,2000,2010,Forest,Forest,4183.88
1,2000,2010,Forest,Rangeland,84.42
2,2000,2010,Forest,Water body,0.02
3,2000,2010,Forest,Cropland,31.90
4,2000,2010,Forest,Built-up,22.58


In [ ]:
# Export transition analysis
TM_2000_2010.to_csv("TransitionMatrix_2000_2010.csv")
TM_2010_2020.to_csv("TransitionMatrix_2010_2020.csv")

CHANGE_2000_2010.to_csv("ChangeStatistics_2000_2010.csv")
CHANGE_2010_2020.to_csv("ChangeStatistics_2010_2020.csv")

Sankey_Data.to_csv("Sankey_Input_Data.csv", index=False)

✓ Transition analysis successfully exported.


**STEP 3 — Sankey Diagram**

In [ ]:
# Install required libraries
!pip -q install plotly kaleido

In [ ]:
# Import libraries
import pandas as pd
import plotly.graph_objects as go

Libraries successfully imported.


In [ ]:
# Read transition table
sankey = pd.read_csv("Sankey_Input_Data.csv")

print("Number of transitions:", len(sankey))

sankey.head()

Number of transitions: 53


,source_year,target_year,source_class,target_class,area_km2
0,2000,2010,Forest,Forest,4183.88
1,2000,2010,Forest,Rangeland,84.42
2,2000,2010,Forest,Water body,0.02
3,2000,2010,Forest,Cropland,31.90
4,2000,2010,Forest,Built-up,22.58


In [ ]:
# Define Sankey node layout
classes = [

    "Forest",
    "Rangeland",
    "Wetland",
    "Water body",
    "Cropland",
    "Built-up"

]

years = [2000, 2010, 2020]

node_ids = []

for year in years:
    for cls in classes:
        node_ids.append(f"{cls}_{year}")

# Labels

node_labels = []

for year in years:
    for cls in classes:
        node_labels.append(cls)

# Fixed X positions
node_x = (

    [0.05] * 6 +      # 2000

    [0.50] * 6 +      # 2010

    [0.95] * 6        # 2020

)

# Fixed Y positions
y_positions = [

    0.05,   # Forest
    0.30,   # Rangeland
    0.50,   # Wetland
    0.65,   # Water body
    0.82,   # Cropland
    0.95    # Built-up

]

node_y = y_positions * 3

# LULC colors
class_colors = {

    "Forest": "#006400",
    "Rangeland": "#4BEB37",
    "Wetland": "#00C8C8",
    "Water body": "#1E88E1",
    "Cropland": "#F4E100",
    "Built-up": "#FF0000"

}

# Node colors
node_colors = []

for year in years:
    for cls in classes:
        node_colors.append(class_colors[cls])

print(f"Total nodes: {len(node_ids)}")

Total nodes: 18


In [ ]:
# Build Sankey links
node_lookup = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

# Source class colors (RGBA with transparency)
flow_color_lookup = {

    "Forest": "rgba(0,100,0,0.45)",

    "Rangeland": "rgba(75,235,55,0.45)",

    "Wetland": "rgba(0,200,200,0.45)",

    "Water body": "rgba(30,136,225,0.45)",

    "Cropland": "rgba(244,225,0,0.45)",

    "Built-up": "rgba(255,0,0,0.45)"

}

# Lists required by Plotly
sources = []

targets = []

values = []

flow_colors = []

for _, row in sankey.iterrows():

    # Internal node IDs
    source_id = f"{row['source_class']}_{int(row['source_year'])}"
    target_id = f"{row['target_class']}_{int(row['target_year'])}"

    # Node indices
    sources.append(node_lookup[source_id])

    targets.append(node_lookup[target_id])

    # Transition area (km²)
    values.append(row["area_km2"])

    # Flow color
    flow_colors.append(
        flow_color_lookup[row["source_class"]]
    )
print(f"Total nodes : {len(node_ids)}")
print(f"Total links : {len(values)}")

Total nodes : 18
Total links : 53


In [ ]:
# Create publication-quality Sankey diagram

fig = go.Figure(

    go.Sankey(

        arrangement="fixed",

  # Nodes
            node=dict(

            pad=100,

            thickness=60,

            line=dict(
                color="black",
                width=1
            ),

            label=node_labels,

            color=node_colors,

            x=node_x,

            y=node_y,

            hovertemplate="%{label}<extra></extra>"

        ),

      # Links
            link=dict(

            source=sources,

            target=targets,

            value=values,

            color=flow_colors,

            hovertemplate=
            "<b>%{source.label}</b> → <b>%{target.label}</b><br>" +
            "Area: %{value:.2f} km²<extra></extra>"

        )

    )

)

✓ Sankey diagram successfully created.


In [ ]:
# Layout, display and save
fig.update_layout(

    width=2244,
    height=1181,
    autosize=False,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Arial",
        size=36,
        color="black"
    ),

    margin=dict(
        l=0,
        r=0,
        t=120,
        b=0
    )
)

# Year labels
year_annotations = [

    (0.03, "2000"),
    (0.50, "2010"),
    (0.97, "2020")

]

for xpos, year in year_annotations:

    fig.add_annotation(

        x=xpos,
        y=1.1,

        xref="paper",
        yref="paper",

        text=f"<b>{year}</b>",

        showarrow=False,

        font=dict(
            family="Arial",
            size=36,
            color="black"
        )

    )

# Display figure
fig.show()